# CreditGuard: Credit-Risk Assessment and Customer Segmentation Pipeline

CreditGuard solves two fundamental financial lending problems using mathematical algorithms implemented **from scratch in NumPy**:
1. **Credit Risk Prediction**: Distinguishing good credit from bad credit applicants (optimizing for **Recall on Bad Credit**).
2. **Customer Segmentation & Loan Sizing**: Divisive graph clustering and EM-GMM customer profiling on PCA-reduced representations.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
sys.path.append(os.path.abspath('..'))

from src.data_loader import get_preprocessed_credit_data
from src.linear_regression import SimpleLinearRegression, MultipleLinearRegression
from src.pca import PCA
from src.decision_tree import DecisionTreeClassifier
from src.adaboost import AdaBoostClassifier
from src.svm import LinearSVM, KernelSVM
from src.gmm import GaussianMixtureModel
from src.graph_clustering import DivisiveGraphClustering

sns.set_theme(style="darkgrid")

## 1. Data Ingestion & Preprocessing (Stratified 80/20 Split)

In [ ]:
data = get_preprocessed_credit_data(test_size=0.2, random_state=42)
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']
X_reg_train, X_reg_test = data['X_reg_train'], data['X_reg_test']
y_reg_train, y_reg_test = data['y_reg_train'], data['y_reg_test']

print(f"X_train shape: {X_train.shape}, y_train bad credit rate: {np.mean(y_train):.1%}")
print(f"X_test shape:  {X_test.shape},  y_test bad credit rate:  {np.mean(y_test):.1%}")

## 2. Loan Amount Regression (Normal Equation & Gradient Descent)

In [ ]:
reg = MultipleLinearRegression(method='normal', l2_reg=1e-5).fit(X_reg_train, y_reg_train)
print(f"Test R^2: {reg.score(X_reg_test, y_reg_test):.4f}")
print(f"Test RMSE: ${reg.rmse(X_reg_test, y_reg_test):,.2f}")

## 3. PCA Eigendecomposition & Risk Projection

In [ ]:
pca = PCA(n_components=2).fit(X_train)
X_pca2 = pca.transform(X_train)
print(f"Explained Variance Ratio (PC1, PC2): {pca.explained_variance_ratio_}")

## 4. Decision Tree Rule Inspection

In [ ]:
dt = DecisionTreeClassifier(max_depth=3).fit(X_train, y_train)
print(dt.export_text(feature_names=data['feature_names'], class_names=['Good Credit', 'Bad Credit']))

## 5. Customer Segmentation (EM-GMM & Divisive Graph Clustering)

In [ ]:
gmm = GaussianMixtureModel(n_components=3, max_iter=50, random_state=42).fit(X_pca2)
gmm_clusters = gmm.predict(X_pca2)

dgc = DivisiveGraphClustering(n_clusters=3, n_neighbors=8).fit(X_pca2[:300])
dgc_clusters = dgc.predict(X_pca2)

print("GMM component weights:", gmm.weights_)
print("Divisive Graph best modularity:", dgc.best_modularity_)